In [1]:
pip install biopython

Note: you may need to restart the kernel to use updated packages.


Modulos Necessarios

In [4]:
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML
from Bio import SeqIO

from collections import Counter

BLASTp das Protainas 

In [3]:
Protainas = [
    "P9A56_gp33_protein.fasta",
    "P9A56_gp34_protein.fasta",
    "P9A56_gp35_protein.fasta"
]

for protein_file in Protainas:
    record = SeqIO.read(protein_file, "fasta")
    result_handle = NCBIWWW.qblast("blastp", "nr", record.seq)

    xml_file = protein_file.replace("_protein.fasta", "_blast.xml")
    with open(xml_file, "w") as out_handle:
        out_handle.write(result_handle.read())
    result_handle.close()


Leitura dos Resultados

O codigo vai selecionar aos 10 melhores resultados e filtrar o e-value

In [5]:
blast_files = [
    "P9A56_gp33_blast.xml",
    "P9A56_gp34_blast.xml",
    "P9A56_gp35_blast.xml"
]

for blast_file in blast_files:

    with open(blast_file) as result_handle:
        blast_records = NCBIXML.parse(result_handle)

        homologs_info = []
        funcoes = []

        for blast_record in blast_records:
            print(f"Query: {blast_record.query}")
            print("Homólogos significativos:")

            for alignment in blast_record.alignments[:10]:
                for hsp in alignment.hsps:
                    if hsp.expect < 1e-5 and hsp.identities / hsp.align_length >= 0.3:
                        print(f"Hit: {alignment.title}")
                        print(f"E-value: {hsp.expect}")
                        print(f"Identidade: {hsp.identities}/{hsp.align_length} ({hsp.identities/hsp.align_length*100:.1f}%)")
                        print("="*40)

                        homologs_info.append({
                            "title": alignment.title,
                            "e_value": hsp.expect,
                            "identidade": hsp.identities,
                            "align_length": hsp.align_length
                        })

                        if "[" in alignment.title:
                            func = alignment.title.split("[")[0].strip()
                        else:
                            func = alignment.title
                        funcoes.append(func)
                        break

    contagem_funcoes = Counter(funcoes)
    print("\nFunções Mais Frequentes entre os Homólogos Significativos:")
    for func, count in contagem_funcoes.most_common(10):
        print(f"{func}: {count}")


Query: unnamed protein product
Homólogos significativos:
Hit: ref|YP_010739261.1| endolysin [Pseudomonas phage phiH2] >ref|YP_010739323.1| endolysin [Pseudomonas phage AAT-1] >ref|YP_010739414.1| endolysin [Pseudomonas phage phiH1] >gb|WGH15499.1| putative endolysin [Pseudomonas phage PA_LZ02] >gb|AME18059.1| endolysin [Pseudomonas phage AAT-1] >gb|UXX42079.1| endolysin [Pseudomonas phage phiH2] >gb|UYD21614.1| endolysin [Pseudomonas phage phiH1]
E-value: 5.25811e-118
Identidade: 166/166 (100.0%)
Hit: ref|YP_009210649.1| endolysin [Pseudomonas phage PaMx28] >gb|ALH23637.1| putative endolysin [Pseudomonas phage PaMx28]
E-value: 5.06201e-102
Identidade: 144/166 (86.7%)
Hit: ref|YP_009199477.1| endolysin [Pseudomonas phage PaMx74] >gb|ALH23507.1| putative endolysin [Pseudomonas phage PaMx74]
E-value: 6.70809e-93
Identidade: 134/165 (81.2%)
Hit: ref|YP_010738480.1| endolysin [Stenotrophomonas phage vB_Sm_QDWS359] >gb|UQM93921.1| endolysin [Stenotrophomonas phage vB_Sm_QDWS359]
E-value: 2.9